In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [ ]:
ridge_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_national_residual_programs.csv")
xgb_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_xgb_national_residual_programs.csv")
ipedsd_df = pd.read_csv(root/"data"/"clean"/"ipeds"/"clean_ipeds_drivers.csv")

In [ ]:
ridge_residual_df['sign_agreement'] = (
    np.sign(ridge_residual_df['1_year_error']) == 
    np.sign(ridge_residual_df['4_year_error'])
) & (
    np.sign(ridge_residual_df['4_year_error']) == 
    np.sign(ridge_residual_df['5_year_error'])
)

print(ridge_residual_df['sign_agreement'].value_counts(normalize=True))

print(ridge_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True     0.61211
False    0.38789
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.798756      0.714188
4_year_error      0.798756      1.000000      0.816227
5_year_error      0.714188      0.816227      1.000000


to avoid canceled signals

In [ ]:
xgb_residual_df['sign_agreement'] = (
    np.sign(xgb_residual_df['1_year_error']) == 
    np.sign(xgb_residual_df['4_year_error'])
) & (
    np.sign(xgb_residual_df['4_year_error']) == 
    np.sign(xgb_residual_df['5_year_error'])
)

print(xgb_residual_df['sign_agreement'].value_counts(normalize=True))

print(xgb_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True    1.0
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.851905      0.799568
4_year_error      0.851905      1.000000      0.841716
5_year_error      0.799568      0.841716      1.000000


In [9]:
ridge_residual_df['total_pred'] = ridge_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

ridge_residual_df["total_count"] = (
    ridge_residual_df["1_yr_working_count"] +
    ridge_residual_df["4_yr_working_count"] +
    ridge_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(ridge_residual_df["total_count"]), 75)

ridge_residual_df["weight"] = (
    np.log1p(ridge_residual_df["total_count"]) /
    np.log1p(ridge_residual_df["total_count"] + k)
)

ridge_residual_df["combined_pct_error"] = ridge_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / ridge_residual_df['total_pred'] * 3

# consistent_mask = ridge_residual_df['sign_agreement'] == True
# ridge_residual_df = ridge_residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(ridge_residual_df)}")

ridge_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,total_pred,total_count,weight,combined_pct_error
0,100,3,110422,1,Public,21.0,CA,35.299513,NaN,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.027100,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,61518.918041,6831.081959,84412.0,11.343465,61849.328337,11.032457,22562.671663,36,64786.0,11.078845,44661.282885,10.706862,20124.717115,18.0,28,high,0.450608,0.418702,0.364801,0.352710,0.111040,0.104505,2.0,2.0,10.0,0.0,8.0,8.0,4.618802,0.171067,0.16765,0.352710,True,168029.529263,75.0,0.983097,0.359307
1,100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.889413,"Agriculture, General.",Delaware State University,47478.0,53605.820876,-6127.820876,52676.0,10.871915,54423.909124,10.904559,-1747.909124,24,38873.0,10.568055,39038.464664,10.572303,-165.464664,22.0,28,high,-0.004239,-0.003998,-0.032117,-0.030432,-0.114313,-0.108515,16.0,15.0,18.0,-1.0,3.0,2.0,1.527525,0.056575,0.16765,-0.030432,True,147068.194664,70.0,0.981691,-0.035655
2,100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.967096,"Agriculture, General.",Illinois State University,64041.0,57936.128323,6104.871677,63600.0,11.060369,59188.291176,10.988479,4411.708824,214,47295.0,10.764160,44425.661994,10.701573,2869.338006,205.0,28,high,0.064587,0.064311,0.074537,0.074227,0.105372,0.104631,11.0,8.0,9.0,-3.0,1.0,-2.0,1.527525,0.056575,0.16765,0.074227,True,161550.081493,551.0,0.998326,0.081926
3,100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.804913,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,49262.255760,13768.744240,57596.0,10.961208,50034.114619,10.820460,7561.885381,47,39700.0,10.589106,38819.886275,10.566688,880.113725,22.0,28,high,0.022672,0.021384,0.151135,0.147450,0.279499,0.264632,13.0,4.0,2.0,-9.0,-2.0,-11.0,5.859465,0.217017,0.16765,0.147450,True,138116.256654,92.0,0.986666,0.164250
4,100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.873760,"Agriculture, General.",Western Illinois University,58204.0,52773.247380,5430.752620,58333.0,10.973923,52459.403970,10.867795,5873.596030,160,48509.0,10.789505,38882.312529,10.568295,9626.687471,149.0,28,high,0.247585,0.246045,0.111965,0.111311,0.102907,0.102258,4.0,5.0,11.0,1.0,6.0,7.0,3.785939,0.140220,0.16765,0.111311,True,144114.963878,454.0,0.997908,0.122269


In [10]:
xgb_residual_df['total_pred'] = xgb_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

xgb_residual_df["total_count"] = (
    xgb_residual_df["1_yr_working_count"] +
    xgb_residual_df["4_yr_working_count"] +
    xgb_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(xgb_residual_df["total_count"]), 75)

xgb_residual_df["weight"] = (
    np.log1p(xgb_residual_df["total_count"]) /
    np.log1p(xgb_residual_df["total_count"] + k)
)

xgb_residual_df["combined_pct_error"] = xgb_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / xgb_residual_df['total_pred'] * 3

# consistent_mask = xgb_residual_df['sign_agreement'] == True
# xgb_residual_df = xgb_residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(xgb_residual_df)}")

xgb_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,sign_agreement,total_pred,total_count,weight,combined_pct_error
0,100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.994006,"Agriculture, General.",Delaware State University,47478.0,59516.336,-12038.335937,52676.0,10.871915,59364.930,10.991459,-6688.929687,24,38873.0,10.568055,40220.414,10.602130,-1347.414063,22.0,28,high,-0.033501,-0.031598,-0.112675,-0.106766,-0.202269,-0.192011,18.0,18.0,24.0,0.0,6.0,6.0,3.464102,0.128300,0.186109,-0.106766,True,159101.680,70.0,0.981356,-0.126126
1,100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.992542,"Agriculture, General.",Illinois State University,64041.0,59429.277,4611.722656,63600.0,11.060369,61214.746,11.022143,2385.253906,214,47295.0,10.764160,42545.203,10.658322,4749.796875,205.0,28,high,0.111641,0.111163,0.038965,0.038803,0.077600,0.077054,8.0,6.0,8.0,-2.0,2.0,0.0,1.154701,0.042767,0.186109,0.077054,True,163189.226,551.0,0.998294,0.084780
2,100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.850296,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,51549.410,11481.589844,57596.0,10.961208,53784.330,10.892737,3811.671875,47,39700.0,10.589106,38761.050,10.565171,938.949219,22.0,28,high,0.024224,0.022848,0.070870,0.069142,0.222730,0.210883,13.0,5.0,1.0,-8.0,-4.0,-12.0,6.110101,0.226300,0.186109,0.069142,True,144094.790,92.0,0.986418,0.079358
3,100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.893735,"Agriculture, General.",Western Illinois University,58204.0,53838.008,4365.992187,58333.0,10.973923,53683.992,10.890870,4649.007813,160,48509.0,10.789505,39219.414,10.576927,9289.585938,149.0,28,high,0.236862,0.235389,0.086600,0.086094,0.081095,0.080584,4.0,4.0,7.0,0.0,3.0,3.0,1.732051,0.064150,0.186109,0.086094,True,146741.414,454.0,0.997868,0.095045
4,100,3,157386,0,Public,60.0,KY,38.186768,NaN,33,13.0,0.7721,35407.0,0.689286,2.0,22.0,1,NaN,NaN,533.0,open,10.692785,10.898738,"Agriculture, General.",Morehead State University,44037.0,54108.030,-10071.031250,45255.0,10.720068,54223.120,10.900863,-8968.121094,91,34670.0,10.453630,38868.105,10.567929,-4198.105469,75.0,28,high,-0.108009,-0.106509,-0.165393,-0.163526,-0.186128,-0.182833,25.0,22.0,23.0,-3.0,1.0,-2.0,1.527525,0.056575,0.186109,-0.163526,True,147199.255,226.0,0.995223,-0.182775


In [ ]:
print("XGBoost residual std:", xgb_residual_df["combined_pct_error"].std())
print("Ridge residual std:  ", ridge_residual_df["combined_pct_error"].std())

XGBoost residual std: 0.17911983646129392
Ridge residual std:   0.18716905228870856


In [12]:
ipedsd_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6909 entries, 0 to 6908
Data columns (total 31 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   6909 non-null   int64  
 1   year                                      6909 non-null   int64  
 2   code                                      6909 non-null   int64  
 3   credential_level                          6909 non-null   int64  
 4   program_completers                        6909 non-null   int64  
 5   program_completers_log                    6909 non-null   float64
 6   program_completer_share_within_school     6906 non-null   float64
 7   instruction_salary_pct                    0 non-null      float64
 8   academic_support_salary_pct               0 non-null      float64
 9   student_services_salary_pct               0 non-null      float64
 10  research_salary_pct                       0 non

In [13]:
ipedsd_df=ipedsd_df.drop(columns=['instruction_salary_pct', 'academic_support_salary_pct',
       'student_services_salary_pct', 'research_salary_pct'])

In [14]:
ipedsd_df.head()

,unit_id,year,code,credential_level,program_completers,program_completers_log,program_completer_share_within_school,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share
0,132374,2020,1101,1,14,2.708050,0.016588,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
1,132374,2020,1102,1,10,2.397895,0.011848,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
2,132374,2020,1108,1,40,3.713572,0.047393,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
3,132374,2020,1109,1,34,3.555348,0.040284,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN
4,132374,2020,1205,1,47,3.871201,0.055687,0.0,0.0,1.0,0.0,1.0,71.0,0.0,5.0,NaN,NaN,0.096838,0.057065,0.057065,0.589286,0.0,0.0,0.0,0.0,NaN,NaN


In [15]:
targ='combined_pct_error'
merge_df=ipedsd_df.merge(ridge_residual_df[["unit_id","code","credential_level","school_name",targ,'weight']], on = ["unit_id","code","credential_level"], how="right")

In [16]:
model_df=merge_df.copy()
print("ipedsd_df shape:", ipedsd_df.shape)
print("ridge_residual_df shape:", ridge_residual_df.shape)
print("model_df shape:", model_df.shape)

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

ipedsd_df shape: (6909, 27)
ridge_residual_df shape: (36219, 63)
model_df shape: (36219, 30)
model_df columns:
['career_counseling', 'code', 'combined_pct_error', 'credential_level', 'employment_services', 'endowment_per_fte', 'equity_ratio', 'instruction_expense_pct', 'instructional_fte_per_student', 'instructional_share_of_staff', 'instructional_staff_long_contract', 'instructional_staff_long_contract_share', 'instructional_staff_short_contract', 'instructional_staff_short_contract_share', 'instructional_staff_total', 'irps_fte_per_student', 'no_ap_credit', 'placement_services', 'program_completer_share_within_school', 'program_completers', 'program_completers_log', 'research_expense_pct', 'research_share_of_staff', 'school_name', 'staff_per_student', 'student_service_expense_pct', 'study_abroad', 'unit_id', 'weight', 'year']


In [17]:
model_df=model_df.drop(columns=[
    'year',
    # 'program_completer_share_within_school','program_completers',
    # 'program_completers_log',"credential_level",
    'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [18]:
inst_model_df=model_df.copy()

In [20]:
display(inst_model_df.describe())
# drop non-numeric columns before correlation
numeric_cols = inst_model_df.select_dtypes(include=[np.number]).columns.tolist()
display("Feature correlation to target variable",
        inst_model_df[numeric_cols].corr()[targ].sort_values())
display(inst_model_df.info())

,unit_id,credential_level,program_completers,program_completers_log,program_completer_share_within_school,no_ap_credit,study_abroad,career_counseling,employment_services,placement_services,instruction_expense_pct,research_expense_pct,student_service_expense_pct,endowment_per_fte,equity_ratio,staff_per_student,instructional_fte_per_student,irps_fte_per_student,instructional_share_of_staff,research_share_of_staff,instructional_staff_total,instructional_staff_short_contract,instructional_staff_long_contract,instructional_staff_long_contract_share,instructional_staff_short_contract_share,combined_pct_error,weight
count,36219.000000,36219.000000,1520.000000,1520.000000,1519.000000,1520.000000,1520.000000,1520.000000,1520.000000,1520.000000,863.000000,863.000000,863.000000,796.000000,814.000000,1520.000000,1520.000000,1520.000000,1520.000000,1520.000000,1520.000000,1520.000000,1520.000000,1374.000000,1374.000000,36219.000000,36219.000000
mean,217548.232557,3.113173,158.026316,4.177508,0.090775,0.096711,0.707895,0.983553,0.935526,0.948026,35.843569,10.667439,7.152955,11190.103015,55.926290,0.114412,0.035464,0.041200,0.366505,0.015273,1347.181579,3.376316,1343.805263,0.994892,0.005108,0.020640,0.992295
std,101742.789118,1.324019,464.151695,1.293150,0.199691,0.295660,0.454880,0.127230,0.245676,0.222047,8.570026,9.613382,4.095129,11172.919693,14.541436,0.099105,0.020159,0.028283,0.119566,0.021403,1475.200164,25.792313,1475.513263,0.030695,0.030695,0.187169,0.005947
min,100654.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,26.000000,0.000000,0.000000,227.000000,37.000000,0.016738,0.007823,0.007823,0.131088,0.000000,0.000000,0.000000,0.000000,0.708333,0.000000,-0.698850,0.971727
25%,154022.000000,2.000000,30.000000,3.433987,0.006235,0.000000,0.000000,1.000000,1.000000,1.000000,30.000000,0.000000,5.000000,2659.000000,43.000000,0.053388,0.021904,0.022601,0.292579,0.000000,176.000000,0.000000,176.000000,1.000000,0.000000,-0.091032,0.988437
50%,195173.000000,3.000000,63.000000,4.158883,0.018008,0.000000,1.000000,1.000000,1.000000,1.000000,36.000000,13.000000,5.000000,7562.000000,57.000000,0.087104,0.032542,0.032542,0.352941,0.000000,680.000000,0.000000,680.000000,1.000000,0.000000,0.000861,0.993731
75%,228723.000000,3.000000,139.000000,4.941642,0.054632,0.000000,1.000000,1.000000,1.000000,1.000000,38.000000,17.000000,10.000000,12079.000000,69.000000,0.143447,0.046802,0.047711,0.431579,0.040203,2636.000000,0.000000,2636.000000,1.000000,0.000000,0.106841,0.997222
max,499705.000000,8.000000,9082.000000,9.114160,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,73.000000,25.000000,18.000000,55596.000000,89.000000,0.937500,0.375000,0.375000,0.833333,0.063864,5044.000000,238.000000,5044.000000,1.000000,0.291667,2.930964,0.999979


'Feature correlation to target variable'

research_expense_pct                       -0.325475
endowment_per_fte                          -0.269195
instructional_staff_total                  -0.252144
instructional_staff_long_contract          -0.251920
research_share_of_staff                    -0.239502
staff_per_student                          -0.153745
irps_fte_per_student                       -0.148124
study_abroad                               -0.102630
instructional_fte_per_student              -0.101980
placement_services                         -0.098027
no_ap_credit                               -0.029095
unit_id                                    -0.023728
instructional_staff_short_contract         -0.009806
instructional_staff_short_contract_share   -0.008119
program_completers_log                      0.007029
instructional_staff_long_contract_share     0.008119
credential_level                            0.008182
program_completers                          0.015809
employment_services                         0.

<class 'pandas.DataFrame'>
RangeIndex: 36219 entries, 0 to 36218
Data columns (total 28 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   36219 non-null  int64  
 1   credential_level                          36219 non-null  int64  
 2   program_completers                        1520 non-null   float64
 3   program_completers_log                    1520 non-null   float64
 4   program_completer_share_within_school     1519 non-null   float64
 5   no_ap_credit                              1520 non-null   float64
 6   study_abroad                              1520 non-null   float64
 7   career_counseling                         1520 non-null   float64
 8   employment_services                       1520 non-null   float64
 9   placement_services                        1520 non-null   float64
 10  instruction_expense_pct                   863

None

In [21]:
import plotly.express as px
px.histogram(inst_model_df[targ])

In [22]:
len(inst_model_df[targ])

36219

In [23]:
inst_model_df[targ].skew()

np.float64(1.8068397446720743)

In [24]:
inst_model_df[targ].describe()

count    36219.000000
mean         0.020640
std          0.187169
min         -0.698850
25%         -0.091032
50%          0.000861
75%          0.106841
max          2.930964
Name: combined_pct_error, dtype: float64

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split, StratifiedKFold
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 42

In [ ]:
feature_cols = [c for c in inst_model_df.columns if c not in ["unit_id", targ, "weight",'credential_level','code','avg_rank_stability','school_name']]

X = inst_model_df[feature_cols]

y = inst_model_df[targ]
# y = y.clip(upper=y.quantile(0.9))

w = inst_model_df["weight"]

X = X.apply(pd.to_numeric, errors="coerce")


X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=RANDOM_STATE
)

num_cols = X_train.columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols)
])

In [46]:
len(y)

36219

In [47]:
X.columns

Index(['program_completers', 'program_completers_log',
       'program_completer_share_within_school', 'no_ap_credit', 'study_abroad',
       'career_counseling', 'employment_services', 'placement_services',
       'instruction_expense_pct', 'research_expense_pct',
       'student_service_expense_pct', 'endowment_per_fte', 'equity_ratio',
       'staff_per_student', 'instructional_fte_per_student',
       'irps_fte_per_student', 'instructional_share_of_staff',
       'research_share_of_staff', 'instructional_staff_total',
       'instructional_staff_short_contract',
       'instructional_staff_long_contract',
       'instructional_staff_long_contract_share',
       'instructional_staff_short_contract_share'],
      dtype='str')

In [48]:
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge())
])

# ridge_model = TransformedTargetRegressor(
#     regressor=ridge_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

ridge_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0, 100.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ridge_grid.fit(
    X_train,
    y_train,
    reg__sample_weight=w_train
)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(X_test)

print("RIDGE")
print("Best params:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, ridge_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, ridge_preds), 4))

# feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
# print(feature_names)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
RIDGE
Best params: {'reg__alpha': 100.0}
Best CV MAE: 0.0749
Test MAE (vs unclipped y_test): 0.0747
Test R2 (vs unclipped y_test): 0.004


In [49]:
from sklearn.linear_model import Lasso


lasso_param_grid = {
    "reg__alpha": np.logspace(-4, 1, 50)
}

lasso_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=10000, random_state=RANDOM_STATE))
])

lasso_grid = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

lasso_grid.fit(
    X_train,
    y_train,
    reg__sample_weight=w_train
)

lasso_best = lasso_grid.best_estimator_
lasso_preds = lasso_best.predict(X_test)

print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, lasso_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, lasso_preds), 4))

feature_names = lasso_best.named_steps["preprocessor"].get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": [f.replace("num__", "") for f in feature_names],
    "coefficient": lasso_best.named_steps["reg"].coef_
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df[coef_df["coefficient"] != 0].to_string(index=False))
print(f"\nFeatures selected: {(coef_df['coefficient'] != 0).sum()} of {len(coef_df)}")
print(f"Lasso alpha chosen: {lasso_best.named_steps['reg'].alpha:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Test MAE (vs unclipped y_test): 0.0747
Test R2 (vs unclipped y_test): 0.0034
                              feature  coefficient
                         equity_ratio     0.002545
                         study_abroad    -0.002213
            instructional_staff_total    -0.001121
                    career_counseling     0.001061
program_completer_share_within_school     0.000473

Features selected: 5 of 23
Lasso alpha chosen: 0.0007


In [ ]:
from sklearn.linear_model import LassoCV

for seed in [0, 7, 13, 21, 99]:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", LassoCV(cv=5, random_state=seed, max_iter=50000))
    ])
    pipe.fit(X_train, y_train, reg__sample_weight=w_train)

    feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
    selected = [
        f.replace("num__", "")
        for f, c in zip(feature_names, pipe.named_steps["reg"].coef_)
        if c != 0
    ]
    print(f"Seed {seed:3d}: {selected}")

Seed   0: ['program_completer_share_within_school', 'study_abroad', 'career_counseling', 'equity_ratio', 'instructional_staff_total']
Seed   7: ['program_completer_share_within_school', 'study_abroad', 'career_counseling', 'equity_ratio', 'instructional_staff_total']
Seed  13: ['program_completer_share_within_school', 'study_abroad', 'career_counseling', 'equity_ratio', 'instructional_staff_total']
Seed  21: ['program_completer_share_within_school', 'study_abroad', 'career_counseling', 'equity_ratio', 'instructional_staff_total']
Seed  99: ['program_completer_share_within_school', 'study_abroad', 'career_counseling', 'equity_ratio', 'instructional_staff_total']


In [53]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])

# enet_model = TransformedTargetRegressor(
#     regressor=enet_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005,0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    estimator=enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(
    X_train, 
    y_train,
    reg__sample_weight=w_train
)

enet_best = enet_grid.best_estimator_
enet_preds = enet_best.predict(X_test)

print("\nELASTIC NET")
print("Best params:", enet_grid.best_params_)
print("Best CV MAE:", round(-enet_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, enet_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

ELASTIC NET
Best params: {'reg__alpha': 0.1, 'reg__l1_ratio': 0.005}
Best CV MAE: 0.0748
Test MAE (vs unclipped y_test): 0.0747
Test R2 (vs unclipped y_test): 0.0034


In [54]:
y_pred = [y.mean()] * len(y)

mean_absolute_error(y, y_pred)

0.07496248990667338

In [55]:
from sklearn.ensemble import HistGradientBoostingRegressor

def run_hgbr():
    hgbr_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
    ])
    

    hgbr_model.fit(
        X_train, 
        y_train,
        model__sample_weight=w_train
    ) 
    preds = hgbr_model.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds)) 
    print("R2:", r2_score(y_test, preds))
run_hgbr()

MAE: 0.07482318303320193
R2: 0.001954038833428484


In [56]:
def run_hgbr_2():
    hgbr_model_2 = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor(
        max_depth=5,              # allow more interactions
        learning_rate=0.01,       # slower learning
        max_iter=750,             # more trees
        min_samples_leaf=10,      # regularization
        l2_regularization=2.0,    # stabilize
        random_state=42
        ))
    ])
  

    hgbr_model_2.fit(
        X_train, 
        y_train,
        model__sample_weight=w_train
    ) 

    preds_2 = hgbr_model_2.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds_2)) 
    print("R2:", r2_score(y_test, preds_2))
run_hgbr_2()

MAE: 0.07485950221121844
R2: 0.0018865624186880092


In [ ]:
from sklearn.model_selection import cross_val_score

y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_hgbr_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
    ])

r2_scores = []

for train_idx, val_idx in cv.split(X, y):
    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]
    w_tr = w.iloc[train_idx]

    model = clone(cv_hgbr_model)
    model.fit(X_tr, y_tr, model__sample_weight=w_tr)

    val_pred = model.predict(X_val)
    r2_scores.append(r2_score(y_val, val_pred))

print(""" \
HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
""")
print("R2 scores:", [round(s, 4) for s in r2_scores])
print("R2 mean:", round(np.mean(r2_scores), 4))
print("R2 std:", round(np.std(r2_scores), 4))

 HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))

R2 scores: [0.0017, 0.0019, 0.0031, 0.0025, 0.0006]
R2 mean: 0.0019
R2 std: 0.0008


In [62]:
y_shuffled = y.sample(frac=1, random_state=42).reset_index(drop=True)

cv_hgbr_model.fit(X_train, y_shuffled.loc[X_train.index], model__sample_weight=w_train)
preds = cv_hgbr_model.predict(X_test)

print("R2 shuffled:", r2_score(y_test, preds))

R2 shuffled: -0.00015335630656321264


In [63]:
hgbr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    ))
])

r2_scores = []

for train_idx, val_idx in cv.split(X, y_bins):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    w_tr = w.iloc[train_idx]

    model = clone(hgbr_pipe)
    model.fit(X_tr, y_tr, model__sample_weight=w_tr)

    val_pred = model.predict(X_val)
    r2_scores.append(r2_score(y_val, val_pred))

print("R2 scores:", [round(s, 4) for s in r2_scores])
print("mean:", round(np.mean(r2_scores), 4))
print("std:", round(np.std(r2_scores), 4))

R2 scores: [0.0017, 0.0019, 0.0031, 0.0025, 0.0006]
mean: 0.0019
std: 0.0008


In [64]:
from xgboost import XGBRegressor

cv_xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42
    ))
])

r2_scores = []

for train_idx, val_idx in cv.split(X, y_bins):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    w_tr = w.iloc[train_idx]

    model = clone(cv_xgb_model)
    model.fit(X_tr, y_tr, model__sample_weight=w_tr)

    val_pred = model.predict(X_val)
    r2_scores.append(r2_score(y_val, val_pred))

print("XGBRegressor")
print("R2 scores:", [round(s, 4) for s in r2_scores])
print("mean:", round(np.mean(r2_scores), 4))
print("std:", round(np.std(r2_scores), 4))

XGBRegressor
R2 scores: [-0.0023, -0.002, 0.0011, 0.0035, -0.0026]
mean: -0.0004
std: 0.0024


In [ ]:
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w,
    test_size=0.2,
    random_state=RANDOM_STATE
)


for obj in [X_train, X_test, y_train, y_test, w_train, w_test]:
    obj.reset_index(drop=True, inplace=True)

y_bins_train = pd.qcut(y_train, q=5, labels=False, duplicates="drop")

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

param_grid_xgb = {
    "reg__n_estimators": [200, 300],
    "reg__max_depth": [5, 10],
    "reg__learning_rate": [0.05, 0.1],
    "reg__subsample": [0.6, 0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring="neg_mean_absolute_error",
    cv=cv.split(X_train, y_bins_train),
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

grid_xgb.fit(X_train, y_train, reg__sample_weight=w_train)

print("XGB Best params:", grid_xgb.best_params_)
print("XGB Best CV MAE:", round(-grid_xgb.best_score_, 2))

best_xgb = grid_xgb.best_estimator_
xgb_preds = best_xgb.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test, xgb_preds), 2))
print("XGB Test R2:", round(r2_score(y_test, xgb_preds), 4))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
XGB Best params: {'reg__learning_rate': 0.05, 'reg__max_depth': 5, 'reg__n_estimators': 200, 'reg__subsample': 0.6}
XGB Best CV MAE: 0.08
XGB Test MAE: 0.08
XGB Test R2: -0.0097


In [ ]:
pipe_hgbr = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
        verbose=0
    ))
])

param_grid_hgbr = {
    'reg__max_iter': [200, 300],
    'reg__max_depth': [5, 10],
    'reg__learning_rate': [0.05, 0.1],
}

grid_hgbr = GridSearchCV(
    estimator=pipe_hgbr,
    param_grid=param_grid_hgbr,
    scoring='neg_mean_absolute_error',
    cv=cv.split(X_train, y_bins_train),
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_hgbr.fit(X_train, y_train, reg__sample_weight=w_train)

print("HGBR Best params:", grid_hgbr.best_params_)
print("HGBR Best CV MAE:", round(-grid_hgbr.best_score_, 2))

best_hgbr = grid_hgbr.best_estimator_
hgbr_preds = best_hgbr.predict(X_test)

print("HGBR Test MAE:", round(mean_absolute_error(y_test, hgbr_preds), 2))
print("HGBR Test R2:", round(r2_score(y_test, hgbr_preds), 4))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
HGBR Best params: {'reg__learning_rate': 0.05, 'reg__max_depth': 5, 'reg__max_iter': 200}
HGBR Best CV MAE: 0.07
HGBR Test MAE: 0.07
HGBR Test R2: 0.0009


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score

y_raw = np.log1p(inst_model_df[targ].abs())
# y_raw = y.clip(upper=y.quantile(0.95))
# y_raw = inst_model_df[targ].copy()

low = y_raw.quantile(0.25)
high = y_raw.quantile(0.75)

mask = (y_raw <= low) | (y_raw >= high)

X_clf = inst_model_df.drop(columns=["unit_id",'school_name', targ, "weight",'credential_level','code','avg_rank_stability'], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)
w_clf = inst_model_df['weight'].loc[mask].copy()

# force everything numeric
X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_clf,
    y_clf,
    w_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", LogisticRegression(max_iter=10000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []

for train_idx, val_idx in cv.split(X_clf, y_clf):
    X_tr = X_clf.iloc[train_idx]
    X_val = X_clf.iloc[val_idx]
    y_tr = y_clf.iloc[train_idx]
    y_val = y_clf.iloc[val_idx]
    w_tr = w_clf.iloc[train_idx]

    model = clone(clf)
    model.fit(X_tr, y_tr, logit__sample_weight=w_tr)

    val_proba = model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, val_proba))

print(""" \
        LogisticRegression(max_iter=10000)
""")
print("AUC scores:", [round(s, 4) for s in auc_scores])
print("AUC mean:", round(np.mean(auc_scores), 4))
print("AUC std:", round(np.std(auc_scores), 4))

         LogisticRegression(max_iter=10000)

AUC scores: [0.5088, 0.5098, 0.509, 0.5177, 0.5205]
AUC mean: 0.5132
AUC std: 0.005


In [74]:
from sklearn.ensemble import HistGradientBoostingClassifier

clf_hgb = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
    ))
])

auc_scores = []

for train_idx, val_idx in cv.split(X_clf, y_clf):
    X_tr = X_clf.iloc[train_idx]
    X_val = X_clf.iloc[val_idx]
    y_tr = y_clf.iloc[train_idx]
    y_val = y_clf.iloc[val_idx]
    w_tr = w_clf.iloc[train_idx]

    model = clone(clf_hgb)
    model.fit(X_tr, y_tr, logit__sample_weight=w_tr)

    val_proba = model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, val_proba))

print(""" \
        HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42
""")
print("AUC scores:", [round(s, 4) for s in auc_scores])
print("AUC mean:", round(np.mean(auc_scores), 4))
print("AUC std:", round(np.std(auc_scores), 4))

         HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42

AUC scores: [0.5101, 0.5104, 0.5088, 0.518, 0.5195]
AUC mean: 0.5134
AUC std: 0.0045


In [75]:
from xgboost import XGBClassifier

xgb_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss"
    ))
])

auc_scores = []

for train_idx, val_idx in cv.split(X_clf, y_clf):
    X_tr = X_clf.iloc[train_idx]
    X_val = X_clf.iloc[val_idx]
    y_tr = y_clf.iloc[train_idx]
    y_val = y_clf.iloc[val_idx]
    w_tr = w_clf.iloc[train_idx]

    model = clone(xgb_clf)
    model.fit(X_tr, y_tr, logit__sample_weight=w_tr)

    val_proba = model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, val_proba))

print(""" \
        XGBClassifier n_estimators=300,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss"
""")
print("AUC scores:", [round(s, 4) for s in auc_scores])
print("AUC mean:", round(np.mean(auc_scores), 4))
print("AUC std:", round(np.std(auc_scores), 4))

         XGBClassifier n_estimators=300,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss"

AUC scores: [0.5125, 0.5109, 0.5106, 0.5156, 0.5216]
AUC mean: 0.5143
AUC std: 0.0041


In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

for C in C_values:
    clf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(C=C, max_iter=10000))
    ])
    
    auc_scores = []
    for train_idx, val_idx in cv.split(X_clf, y_clf):
        X_tr, X_val = X_clf.iloc[train_idx], X_clf.iloc[val_idx]
        y_tr, y_val = y_clf.iloc[train_idx], y_clf.iloc[val_idx]
        w_tr = w_clf.iloc[train_idx]
        
        clone(clf).fit(X_tr, y_tr, logit__sample_weight=w_tr)
        model = clone(clf)
        model.fit(X_tr, y_tr, logit__sample_weight=w_tr)
        auc_scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
    
    print(f"C={C}: AUC={np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

C=0.001: AUC=0.5129 ± 0.0044
C=0.01: AUC=0.5132 ± 0.0050
C=0.1: AUC=0.5134 ± 0.0050
C=1: AUC=0.5132 ± 0.0050
C=10: AUC=0.5132 ± 0.0050
C=100: AUC=0.5132 ± 0.0050


In [ ]:
import optuna

def objective(trial):
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    C = trial.suggest_float("C", 1e-3, 100.0, log=True)
    solver = "liblinear" if penalty == "l1" else trial.suggest_categorical(
        "solver_l2", ["lbfgs", "liblinear"]
    )

    clf = Pipeline([
        ("preprocessor", preprocessor),
        ("logit", LogisticRegression(
            penalty=penalty,
            C=C,
            solver=solver,
            class_weight=class_weight,
            max_iter=10000,
            random_state=42
        ))
    ])

    auc_scores = []
    for train_idx, val_idx in cv.split(X_clf, y_clf):
        X_tr, X_val = X_clf.iloc[train_idx], X_clf.iloc[val_idx]
        y_tr, y_val = y_clf.iloc[train_idx], y_clf.iloc[val_idx]
        w_tr = w_clf.iloc[train_idx]

        model = clone(clf)
        model.fit(X_tr, y_tr, logit__sample_weight=w_tr)
        auc_scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))

    return float(np.mean(auc_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

print("Best AUC:", study.best_value)
print("Params:", study.best_params)

[I 2026-04-11 18:50:28,815] A new study created in memory with name: no-name-f841a6ab-78d9-40d9-97d3-71081cec9990
c:\Users\sebas\miniconda3\envs\institutional-roi\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning:

'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.

c:\Users\sebas\miniconda3\envs\institutional-roi\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning:

'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.

c:\Users\sebas\miniconda3\envs\institutional-roi\Lib\site-packages\sklearn

Best AUC: 0.5134877021551529
Params: {'penalty': 'l2', 'class_weight': 'balanced', 'C': 0.05642233371900526, 'solver_l2': 'lbfgs'}


In [ ]:
def objective(trial):
    
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "max_iter": trial.suggest_int("max_iter", 100, 500),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 5.0),
        "random_state": 42
    }
    
    clf_hgb_o = Pipeline([
        ("preprocessor", preprocessor),
        ("hgbr", HistGradientBoostingClassifier(**params))
    ])
  
    
    val_scores = []
    train_scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr = w_train.iloc[train_idx]

        m = clone(clf_hgb_o)
        m.fit(X_tr, y_tr, hgbr__sample_weight=w_tr)
        
        val_scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        train_scores.append(roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1]))

    mean_val = np.mean(val_scores)
    mean_train = np.mean(train_scores)
    gap = mean_train - mean_val


    return mean_val - 0.5 * gap

hgbr_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
hgbr_study.optimize(objective, n_trials=100)

print("Best trial:")
print("CV AUC:", hgbr_study.best_value)
print("Params:", hgbr_study.best_params)

best_hgbr_params = hgbr_study.best_params.copy()
best_hgbr_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
})


best_hgbr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(**best_hgbr_params))
])

best_hgbr.fit(X_train, y_train, model__sample_weight=w_train)

hgbr_train_proba = best_hgbr.predict_proba(X_train)[:, 1]
hgbr_test_proba = best_hgbr.predict_proba(X_test)[:, 1]

[I 2026-04-11 18:52:41,967] A new study created in memory with name: no-name-6e1fe118-5278-48b8-94ca-eed7ed149b1b
[I 2026-04-11 18:52:42,924] Trial 0 finished with value: 0.5091212149099118 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 6, 'max_iter': 393, 'min_samples_leaf': 64, 'l2_regularization': 0.7800932022121826}. Best is trial 0 with value: 0.5091212149099118.
[I 2026-04-11 18:52:43,817] Trial 1 finished with value: 0.5088657262185527 and parameters: {'learning_rate': 0.015957084694148364, 'max_depth': 2, 'max_iter': 447, 'min_samples_leaf': 64, 'l2_regularization': 3.540362888980227}. Best is trial 0 with value: 0.5091212149099118.
[I 2026-04-11 18:52:45,453] Trial 2 finished with value: 0.5023417210512179 and parameters: {'learning_rate': 0.010636066512540286, 'max_depth': 6, 'max_iter': 433, 'min_samples_leaf': 29, 'l2_regularization': 0.9091248360355031}. Best is trial 0 with value: 0.5091212149099118.
[I 2026-04-11 18:52:46,535] Trial 3 finished with 

Best trial:
CV AUC: 0.5116300880799295
Params: {'learning_rate': 0.01057087102063208, 'max_depth': 4, 'max_iter': 100, 'min_samples_leaf': 72, 'l2_regularization': 0.6413850149278265}


c:\Users\sebas\miniconda3\envs\institutional-roi\Lib\site-packages\xgboost\core.py:160: UserWarning:

[18:54:15] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\learner.cc:742: 
Parameters: { "l2_regularization", "max_iter", "min_samples_leaf" } are not used.




In [ ]:
from sklearn.metrics import log_loss

hgbr_train_pred = (hgbr_train_proba >= 0.434).astype(int)
hgbr_test_pred = (hgbr_test_proba >= 0.434).astype(int)

train_auc = roc_auc_score(y_train, hgbr_train_proba, sample_weight=w_train)
test_auc = roc_auc_score(y_test, hgbr_test_proba, sample_weight=w_test)

train_acc = accuracy_score(y_train, hgbr_train_pred)
test_acc = accuracy_score(y_test, hgbr_test_pred)

train_logloss = log_loss(y_train, hgbr_train_proba, sample_weight=w_train)
test_logloss = log_loss(y_test, hgbr_test_proba, sample_weight=w_test)

print("\nHGBClassifier")
print("Train AUC:", round(train_auc, 4))
print("Test AUC:", round(test_auc, 4))
print("Train Accuracy:", round(train_acc, 4))
print("Test Accuracy:", round(test_acc, 4))
print("Train LogLoss:", round(train_logloss, 4))
print("Test LogLoss:", round(test_logloss, 4))

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, hgbr_test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, hgbr_test_pred, digits=4))


HGBClassifier
Train AUC: 0.5191
Test AUC: 0.5116
Train Accuracy: 0.5057
Test Accuracy: 0.5038
Train LogLoss: 0.6885
Test LogLoss: 0.6913

Confusion Matrix (Test):
[[  39 2678]
 [  18 2698]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     0.6842    0.0144    0.0281      2717
           1     0.5019    0.9934    0.6668      2716

    accuracy                         0.5038      5433
   macro avg     0.5930    0.5039    0.3475      5433
weighted avg     0.5931    0.5038    0.3474      5433



In [ ]:
from sklearn.metrics import average_precision_score, f1_score

hgbc_train_ap = average_precision_score(y_train, hgbr_train_proba, sample_weight=w_train)
hgbc_test_ap = average_precision_score(y_test, hgbr_test_proba, sample_weight=w_test)
hgbc_train_f1 = f1_score(y_train, hgbr_train_pred, sample_weight=w_train)
hgbc_test_f1 = f1_score(y_test, hgbr_test_pred, sample_weight=w_test)

print("Train AP:", round(hgbc_train_ap, 4))
print("Test AP:", round(hgbc_test_ap, 4))
print("Train F1:", round(hgbc_train_f1, 4))
print("Test F1:", round(hgbc_test_f1, 4))

Train AP: 0.5186
Test AP: 0.5083
Train F1: 0.6681
Test F1: 0.6667


In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, f1_score, precision_score, recall_score

thresholds = np.linspace(0.01, 0.99, 200)

metrics = {
    "f1": [], "precision": [], "recall": [], "accuracy": []
}

for t in thresholds:
    hgbc_preds = (hgbr_train_proba >= t).astype(int)
    metrics["f1"].append(f1_score(y_train, hgbc_preds, zero_division=0))
    metrics["precision"].append(precision_score(y_train, hgbc_preds, zero_division=0))
    metrics["recall"].append(recall_score(y_train, hgbc_preds, zero_division=0))
    metrics["accuracy"].append(accuracy_score(y_train, hgbc_preds))

best_idx = np.argmax(metrics["f1"])
hgbc_best_thresh = thresholds[best_idx]

fig = go.Figure()

fig.add_trace(go.Scatter(x=thresholds, y=metrics["f1"],
    name="F1", line=dict(color="#636EFA", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["precision"],
    name="Precision", line=dict(color="#EF553B", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["recall"],
    name="Recall", line=dict(color="#00CC96", width=2)))
fig.add_trace(go.Scatter(x=thresholds, y=metrics["accuracy"],
    name="Accuracy", line=dict(color="#FFA15A", width=2)))

fig.add_vline(
    x=hgbc_best_thresh,
    line_dash="dash",
    line_color="white",
    annotation_text=f"Best F1 threshold: {hgbc_best_thresh:.3f}",
    annotation_position="top right"
)

fig.update_layout(
    title="Threshold vs Classification Metrics (Train)",
    xaxis_title="Threshold",
    yaxis_title="Score",
    legend=dict(orientation="h", y=-0.15),
    template="plotly_dark",
    hovermode="x unified"
)

fig.show()
print(f"\nBest F1 threshold: {hgbc_best_thresh:.4f}")
print(f"At this threshold — F1: {metrics['f1'][best_idx]:.4f} | "
      f"Precision: {metrics['precision'][best_idx]:.4f} | "
      f"Recall: {metrics['recall'][best_idx]:.4f}")

# apply to test
hgbc_test_pred_tuned = (hgbr_test_proba >= hgbc_best_thresh).astype(int)
print(f"\nTest set with tuned threshold ({hgbc_best_thresh:.3f}):")
print(confusion_matrix(y_test, hgbc_test_pred_tuned))
print(classification_report(y_test, hgbc_test_pred_tuned, digits=4))


Best F1 threshold: 0.4335
At this threshold — F1: 0.6684 | Precision: 0.5029 | Recall: 0.9964

Test set with tuned threshold (0.434):
[[  39 2678]
 [  18 2698]]
              precision    recall  f1-score   support

           0     0.6842    0.0144    0.0281      2717
           1     0.5019    0.9934    0.6668      2716

    accuracy                         0.5038      5433
   macro avg     0.5930    0.5039    0.3475      5433
weighted avg     0.5931    0.5038    0.3474      5433

